# 06 — Treinamento do MLP

**Projeto:** Medical Triage MLOps

## Objetivo

Treinar o primeiro modelo próprio do projeto para classificar os abstracts em:

```text
normal
atenção
urgente
```

O pipeline será:

```text
medical_abstract
      ↓
TF-IDF
      ↓
MLPClassifier
      ↓
normal / atenção / urgente
```

A validação será usada para avaliar o baseline antes de consultar o conjunto de teste.

> **Importante:** os targets utilizados são pseudo-rótulos produzidos a partir do BioBERT e da regra de threshold definida nos notebooks anteriores. Portanto, as métricas deste notebook medem a capacidade do MLP de aprender esses pseudo-rótulos, e não validade clínica real.

## 1. Dependências

Caso ainda não estejam instaladas:

```bash
uv add scikit-learn joblib matplotlib pandas numpy
uv sync
```

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.20,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)

RANDOM_STATE = 42

CLASS_ORDER = ["normal", "atenção", "urgente"]

COLORS = {
    "normal": "#16A085",
    "atenção": "#F4B942",
    "urgente": "#D63031",
}

## 2. Carregamento dos splits

In [ ]:
def locate_project_root() -> Path:
    current = Path.cwd().resolve()

    for candidate in [current, current.parent, current.parent.parent]:
        expected = candidate / "data" / "processed" / "splits" / "train.csv"
        if expected.exists():
            return candidate

    raise FileNotFoundError(
        "Não encontrei data/processed/splits/train.csv. "
        "Execute primeiro o notebook 05_data_split.ipynb."
    )


PROJECT_ROOT = locate_project_root()
SPLITS_DIR = PROJECT_ROOT / "data" / "processed" / "splits"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "docs" / "results"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
validation_df = pd.read_csv(SPLITS_DIR / "validation.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

print(f"Treino:    {len(train_df):,}")
print(f"Validação: {len(validation_df):,}")
print(f"Teste:     {len(test_df):,}")

### Conferência rápida dos dados

In [ ]:
display(
    pd.DataFrame(
        {
            "conjunto": ["Treino", "Validação", "Teste"],
            "registros": [
                len(train_df),
                len(validation_df),
                len(test_df),
            ],
            "textos_únicos": [
                train_df["medical_abstract"].nunique(),
                validation_df["medical_abstract"].nunique(),
                test_df["medical_abstract"].nunique(),
            ],
            "classes": [
                train_df["triage_level"].nunique(),
                validation_df["triage_level"].nunique(),
                test_df["triage_level"].nunique(),
            ],
        }
    )
)

## 3. Variáveis do modelo

O modelo usa exclusivamente:

```text
X = medical_abstract
y = triage_level
```

Os scores do BioBERT não entram como features.

In [ ]:
X_train = train_df["medical_abstract"].astype(str)
y_train = train_df["triage_level"].astype(str)

X_validation = validation_df["medical_abstract"].astype(str)
y_validation = validation_df["triage_level"].astype(str)

X_test = test_df["medical_abstract"].astype(str)
y_test = test_df["triage_level"].astype(str)

print("Classes no treino:")
print(y_train.value_counts().reindex(CLASS_ORDER))

## 4. Pipeline TF-IDF + MLP

### TF-IDF

A representação TF-IDF transforma os textos em vetores numéricos ponderando termos conforme sua frequência no documento e no corpus.

Configuração inicial:

- unigramas e bigramas;
- remoção de stopwords em inglês;
- termos presentes em pelo menos 2 documentos;
- máximo de 5.000 features;
- `sublinear_tf=True`.

### MLP

Usaremos um MLP pequeno como baseline:

```text
entrada TF-IDF
     ↓
64 neurônios
     ↓
3 classes
```

O objetivo neste momento é obter um modelo simples, leve e fácil de comparar com os modelos dos colegas.

In [ ]:
pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                max_features=5_000,
                sublinear_tf=True,
            ),
        ),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(64,),
                activation="relu",
                solver="adam",
                alpha=1e-4,
                batch_size="auto",
                learning_rate_init=1e-3,
                max_iter=200,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=12,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

display(pipeline)

## 5. Treinamento

In [ ]:
training_start = time.perf_counter()

pipeline.fit(
    X_train,
    y_train,
)

training_time = time.perf_counter() - training_start

mlp = pipeline.named_steps["mlp"]
tfidf = pipeline.named_steps["tfidf"]

print(f"Tempo de treinamento: {training_time:.2f} segundos")
print(f"Épocas executadas: {mlp.n_iter_}")
print(f"Features TF-IDF: {len(tfidf.get_feature_names_out()):,}")

### Curva de perda

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    range(1, len(mlp.loss_curve_) + 1),
    mlp.loss_curve_,
    linewidth=2,
    color="#5B5FDE",
)

ax.set_title("Curva de perda durante o treinamento")
ax.set_xlabel("Época")
ax.set_ylabel("Loss")

plt.tight_layout()
plt.show()

## 6. Função de avaliação

Como a classe `urgente` é menor, não vamos olhar apenas para accuracy.

As métricas principais são:

- accuracy;
- balanced accuracy;
- macro precision;
- macro recall;
- macro F1-score;
- métricas por classe.

In [ ]:
def evaluate_model(
    model,
    X,
    y_true,
    dataset_name: str,
):
    y_pred = model.predict(X)

    metrics = {
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "recall_macro": recall_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    }

    return metrics, y_pred

## 7. Avaliação no conjunto de validação

In [ ]:
validation_metrics, y_validation_pred = evaluate_model(
    pipeline,
    X_validation,
    y_validation,
    "validation",
)

display(
    pd.DataFrame([validation_metrics])
)

### Relatório por classe

In [ ]:
validation_report = classification_report(
    y_validation,
    y_validation_pred,
    labels=CLASS_ORDER,
    output_dict=True,
    zero_division=0,
)

validation_report_df = (
    pd.DataFrame(validation_report)
    .T
)

display(validation_report_df)

### Matriz de confusão — validação

In [ ]:
validation_cm = confusion_matrix(
    y_validation,
    y_validation_pred,
    labels=CLASS_ORDER,
)

fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(
    validation_cm,
    cmap="Blues",
)

ax.set_title("Matriz de confusão — validação")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")

ax.set_xticks(range(len(CLASS_ORDER)))
ax.set_yticks(range(len(CLASS_ORDER)))
ax.set_xticklabels(CLASS_ORDER)
ax.set_yticklabels(CLASS_ORDER)

for row in range(len(CLASS_ORDER)):
    for col in range(len(CLASS_ORDER)):
        ax.text(
            col,
            row,
            validation_cm[row, col],
            ha="center",
            va="center",
            fontweight="bold",
        )

fig.colorbar(image, ax=ax)

plt.tight_layout()
plt.show()

## 8. Recall da classe urgente

Para o contexto do projeto, é especialmente importante observar quantos casos pseudo-rotulados como `urgente` o modelo consegue recuperar.

In [ ]:
urgent_row = validation_report_df.loc["urgente"]

urgent_validation = pd.DataFrame(
    {
        "métrica": [
            "precision",
            "recall",
            "f1-score",
            "support",
        ],
        "valor": [
            urgent_row["precision"],
            urgent_row["recall"],
            urgent_row["f1-score"],
            urgent_row["support"],
        ],
    }
)

display(urgent_validation)

## 9. Decisão antes do teste

O conjunto de teste deve funcionar como avaliação final.

Neste notebook, depois de observar a validação, vamos manter **a mesma configuração** do pipeline e fazer uma única avaliação no teste.

Se você decidir alterar arquitetura, `max_features`, `alpha` ou outros hiperparâmetros depois de olhar a validação, faça a alteração **antes** de executar a seção de teste.

## 10. Avaliação final no teste

In [ ]:
test_metrics, y_test_pred = evaluate_model(
    pipeline,
    X_test,
    y_test,
    "test",
)

display(
    pd.DataFrame([test_metrics])
)

### Relatório por classe — teste

In [ ]:
test_report = classification_report(
    y_test,
    y_test_pred,
    labels=CLASS_ORDER,
    output_dict=True,
    zero_division=0,
)

test_report_df = pd.DataFrame(test_report).T

display(test_report_df)

### Matriz de confusão — teste

In [ ]:
test_cm = confusion_matrix(
    y_test,
    y_test_pred,
    labels=CLASS_ORDER,
)

fig, ax = plt.subplots(figsize=(7, 6))

image = ax.imshow(
    test_cm,
    cmap="Blues",
)

ax.set_title("Matriz de confusão — teste")
ax.set_xlabel("Classe prevista")
ax.set_ylabel("Classe real")

ax.set_xticks(range(len(CLASS_ORDER)))
ax.set_yticks(range(len(CLASS_ORDER)))
ax.set_xticklabels(CLASS_ORDER)
ax.set_yticklabels(CLASS_ORDER)

for row in range(len(CLASS_ORDER)):
    for col in range(len(CLASS_ORDER)):
        ax.text(
            col,
            row,
            test_cm[row, col],
            ha="center",
            va="center",
            fontweight="bold",
        )

fig.colorbar(image, ax=ax)

plt.tight_layout()
plt.show()

## 11. Comparação validação x teste

In [ ]:
metrics_comparison = pd.DataFrame(
    [
        validation_metrics,
        test_metrics,
    ]
).set_index("dataset")

display(metrics_comparison)

In [ ]:
plot_metrics = [
    "accuracy",
    "balanced_accuracy",
    "f1_macro",
]

ax = (
    metrics_comparison[plot_metrics]
    .T
    .plot(
        kind="bar",
        figsize=(9, 5),
    )
)

ax.set_title("Validação x teste")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_xlabel("")
ax.legend(frameon=False)

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Latência de inferência

Além da qualidade do modelo, o desafio também envolve latência.

Aqui medimos a previsão **texto a texto**, incluindo:

```text
TF-IDF + MLP
```

Isso se aproxima mais da forma como a API fará uma requisição individual.

In [ ]:
latencies_ms = []

for text in X_test:
    start = time.perf_counter()

    pipeline.predict([text])

    elapsed_ms = (
        time.perf_counter() - start
    ) * 1_000

    latencies_ms.append(elapsed_ms)

latency_metrics = {
    "latency_mean_ms": float(np.mean(latencies_ms)),
    "latency_median_ms": float(np.median(latencies_ms)),
    "latency_p95_ms": float(np.percentile(latencies_ms, 95)),
    "latency_max_ms": float(np.max(latencies_ms)),
}

display(
    pd.DataFrame(
        latency_metrics.items(),
        columns=["métrica", "valor_ms"],
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(
    latencies_ms,
    bins=20,
    color="#5B5FDE",
    edgecolor="white",
    alpha=0.9,
)

ax.axvline(
    latency_metrics["latency_p95_ms"],
    color="#D63031",
    linestyle="--",
    linewidth=2,
    label=f"P95 = {latency_metrics['latency_p95_ms']:.2f} ms",
)

ax.set_title("Latência de inferência no conjunto de teste")
ax.set_xlabel("Milissegundos por texto")
ax.set_ylabel("Quantidade")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

## 13. Exemplos de predição

Essa etapa ajuda a verificar o formato que depois será usado pela FastAPI.

In [ ]:
example_indices = list(
    range(min(5, len(test_df)))
)

examples = test_df.loc[
    example_indices,
    ["medical_abstract", "triage_level"],
].copy()

examples["prediction"] = pipeline.predict(
    examples["medical_abstract"]
)

probabilities = pipeline.predict_proba(
    examples["medical_abstract"]
)

for class_index, class_name in enumerate(pipeline.classes_):
    examples[f"prob_{class_name}"] = probabilities[:, class_index]

display(examples)

## 14. Salvamento do modelo

Salvamos o **pipeline completo**, incluindo TF-IDF e MLP.

Isso permite fazer futuramente:

```python
model = joblib.load(...)
model.predict(["novo laudo"])
```

sem precisar carregar o vectorizer separadamente.

In [ ]:
MODEL_PATH = MODELS_DIR / "mlp_tfidf.joblib"

joblib.dump(
    pipeline,
    MODEL_PATH,
)

model_size_mb = (
    MODEL_PATH.stat().st_size
    / 1024**2
)

print(f"Modelo salvo em: {MODEL_PATH}")
print(f"Tamanho: {model_size_mb:.2f} MB")

## 15. Salvamento das métricas

In [ ]:
RESULTS_PATH = RESULTS_DIR / "mlp_metrics.json"

results = {
    "model": "TF-IDF + MLPClassifier",
    "training_time_seconds": float(training_time),
    "training_iterations": int(mlp.n_iter_),
    "tfidf_features": int(len(tfidf.get_feature_names_out())),
    "model_size_mb": float(model_size_mb),
    "validation": {
        key: float(value)
        for key, value in validation_metrics.items()
        if key != "dataset"
    },
    "test": {
        key: float(value)
        for key, value in test_metrics.items()
        if key != "dataset"
    },
    "latency": latency_metrics,
    "configuration": {
        "hidden_layer_sizes": [64],
        "max_features": 5000,
        "ngram_range": [1, 2],
        "confidence_target_origin": "BioBERT pseudo-labeling + uncertainty zone",
        "random_state": RANDOM_STATE,
    },
}

with open(
    RESULTS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Métricas salvas em: {RESULTS_PATH}")

## 16. Resumo automático do baseline

In [ ]:
urgent_test_recall = (
    test_report_df.loc["urgente", "recall"]
)

summary = f'''
### Resultado do MLP baseline

- **Accuracy no teste:** {test_metrics["accuracy"]:.3f}
- **Balanced accuracy:** {test_metrics["balanced_accuracy"]:.3f}
- **Macro F1:** {test_metrics["f1_macro"]:.3f}
- **Macro recall:** {test_metrics["recall_macro"]:.3f}
- **Recall da classe urgente:** {urgent_test_recall:.3f}
- **Latência média:** {latency_metrics["latency_mean_ms"]:.2f} ms
- **Latência P95:** {latency_metrics["latency_p95_ms"]:.2f} ms
- **Tamanho do modelo:** {model_size_mb:.2f} MB
- **Tempo de treinamento:** {training_time:.2f} s
'''

display(Markdown(summary))

## 17. Conclusão

Ao final deste notebook temos:

```text
models/
└── mlp_tfidf.joblib

docs/results/
└── mlp_metrics.json
```

O arquivo `.joblib` contém o pipeline completo:

```text
texto
  ↓
TF-IDF
  ↓
MLP
  ↓
normal / atenção / urgente
```

### Próximos passos

1. registrar as métricas do MLP para comparação com os modelos dos colegas;
2. decidir qual modelo seguirá para a aplicação;
3. integrar o modelo escolhido à FastAPI;
4. instrumentar a API com métricas do Prometheus;
5. criar o stack Prometheus + Grafana.

### Limitação

Como o MLP foi treinado com pseudo-rótulos, um bom resultado significa que ele conseguiu reproduzir o padrão desses rótulos. Isso não equivale a comprovar segurança ou validade clínica da triagem.